In [1]:

import os
import json
import shutil
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader


In [2]:
############################################################
# Step 1: Rename videos from {id}.mp4 to {gloss}_{id}_{instance_id}.mp4
############################################################

json_file_path = 'WLASL_100.json'   # JSON metadata
video_folder_path = 'raw_videos_mp4' # Folder containing original {id}.mp4
with open(json_file_path, 'r') as file:
    data = json.load(file)

# Rename videos
for gloss_entry in data:
    gloss = gloss_entry["gloss"]
    for instance in gloss_entry["instances"]:
        video_id = instance["video_id"]
        instance_id = instance["instance_id"]
        old_file_name = f"{video_id}.mp4"
        old_file_path = os.path.join(video_folder_path, old_file_name)

        new_file_name = f"{gloss}_{video_id}_{instance_id}.mp4"
        new_file_path = os.path.join(video_folder_path, new_file_name)

        if os.path.exists(old_file_path):
            os.rename(old_file_path, new_file_path)
        else:
            print(f"File not found: {old_file_name}")

File not found: 65225.mp4
File not found: 68011.mp4
File not found: 68208.mp4
File not found: 68012.mp4
File not found: 70212.mp4
File not found: 70266.mp4
File not found: 07085.mp4
File not found: 07086.mp4
File not found: 07087.mp4
File not found: 07088.mp4
File not found: 07089.mp4
File not found: 07090.mp4
File not found: 07091.mp4
File not found: 07092.mp4
File not found: 07093.mp4
File not found: 07094.mp4
File not found: 07095.mp4
File not found: 07096.mp4
File not found: 07097.mp4
File not found: 07098.mp4
File not found: 07099.mp4
File not found: 07071.mp4
File not found: 07072.mp4
File not found: 07073.mp4
File not found: 67424.mp4
File not found: 07075.mp4
File not found: 07076.mp4
File not found: 07077.mp4
File not found: 07078.mp4
File not found: 07079.mp4
File not found: 07080.mp4
File not found: 07081.mp4
File not found: 07082.mp4
File not found: 07083.mp4
File not found: 07084.mp4
File not found: 65539.mp4
File not found: 70173.mp4
File not found: 68538.mp4
File not fou

In [3]:
############################################################
# Step 2: Filter classes with <5 videos and prepare for stratified splitting
############################################################

# Build dictionary: {gloss: [(video_id, instance_id, path)]}
gloss_dict = {}
for filename in os.listdir(video_folder_path):
    if filename.endswith(".mp4"):
        gloss = filename.split("_")[0]
        # video_id and instance_id from the filename if needed
        # but we only need them for naming. The full filename is enough here.
        if gloss not in gloss_dict:
            gloss_dict[gloss] = []
        gloss_dict[gloss].append(filename)

# Filter out classes with fewer than 5 videos
filtered_gloss_dict = {g: v for g, v in gloss_dict.items() if len(v) >= 5}

all_files = []
all_labels = []
for g, vids in filtered_gloss_dict.items():
    for vid in vids:
        all_files.append(vid)
        all_labels.append(g)

all_labels = np.array(all_labels)

# Ensure output folders
output_folder = 'stratified_data'
train_folder = os.path.join(output_folder, "train_data")
val_folder = os.path.join(output_folder, "val_data")
test_folder = os.path.join(output_folder, "test_data")

os.makedirs(train_folder, exist_ok=True)
os.makedirs(val_folder, exist_ok=True)
os.makedirs(test_folder, exist_ok=True)

In [4]:
############################################################
# Step 3: Stratified split into Train/Val/Test
############################################################

# 60% train, 20% val, 20% test
train_files, temp_files, train_labels, temp_labels = train_test_split(
    all_files, all_labels, test_size=0.4, random_state=42, stratify=all_labels
)

val_files, test_files, val_labels, test_labels = train_test_split(
    temp_files, temp_labels, test_size=0.5, random_state=42, stratify=temp_labels
)

# Move files according to splits
def move_files(file_list, labels, dest_folder):
    for f in file_list:
        old_path = os.path.join(video_folder_path, f)
        new_path = os.path.join(dest_folder, f)
        shutil.copy2(old_path, new_path)

move_files(train_files, train_labels, train_folder)
move_files(val_files, val_labels, val_folder)
move_files(test_files, test_labels, test_folder)

In [5]:
############################################################
# Step 4: Feature Extraction (ResNet) on each video sequence
############################################################

# Functions for frame extraction and feature extraction
def extract_frames(video_path, fps, frame_start=0, frame_end=-1):
    cap = cv2.VideoCapture(video_path)
    frames = []
    current_frame = 0

    if not cap.isOpened():
        print(f"Error: Could not open {video_path}")
        return []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if current_frame >= frame_start:
            frames.append(frame)
            if frame_end != -1 and current_frame >= frame_end:
                break
        current_frame += 1
    cap.release()
    return frames

def downsample_or_upsample_frames(frames, current_fps, target_fps=25):
    if current_fps == target_fps:
        return frames
    if current_fps < target_fps:
        factor = target_fps // current_fps
        new_frames = []
        for f in frames:
            new_frames.extend([f]*factor)
        return new_frames
    else:
        factor = current_fps // target_fps
        return frames[::factor]

In [13]:
unique_classes = np.load('unique_classes.npy', allow_pickle=True)
unique_classes = list(unique_classes[0])

In [14]:
def create_sequences(frames, sequence_length=32):
    sequences = []
    for i in range(len(frames)-sequence_length+1):
        sequences.append(frames[i:i+sequence_length])
    return sequences

# Load a pretrained ResNet model for feature extraction
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
resnet_model = models.resnet50(pretrained=True)
resnet_model = nn.Sequential(*list(resnet_model.children())[:-1])
resnet_model.eval()
resnet_model.to(device)

transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std=[0.229,0.224,0.225])
])

def extract_resnet_features(frames, model, transform, device):
    # frames: list of cv2 images
    # returns (seq_length, 2048)
    tensors = torch.stack([transform(f) for f in frames]).to(device)
    with torch.no_grad():
        out = model(tensors) # (seq_length,2048,1,1)
    out = out.view(out.size(0), -1) # (seq_length, 2048)
    return out.cpu().numpy()

def process_split(split_folder):
    # Extract features and labels
    features_list = []
    labels_list = []

    # We know labels from filename (gloss_{video_id}_{instance_id}.mp4)
    # Let's gather all files
    files = [f for f in os.listdir(split_folder) if f.endswith('.mp4')]

    # Label encode all classes based on training data only
    # We have already train_labels etc. from before:
    global label_encoder
    # If not defined, define label_encoder:
    # Fit only on train_labels:
    # NOTE: This is a minimal fix since we had train_labels from splitting stage
    # ensure train_labels is global or captured before calling
    label_encoder = LabelEncoder()
    label_encoder.fit(unique_classes) 

    for f in files:
        path = os.path.join(split_folder, f)
        gloss = f.split('_')[0]
        # Assume 25 fps and full video frames, or from JSON if needed
        # For now let's assume frame extraction w/o start-end from JSON
        # Just extract full video:
        frames = extract_frames(path, fps=25)
        frames = downsample_or_upsample_frames(frames, current_fps=25, target_fps=25)
        seqs = create_sequences(frames, sequence_length=32)

        if len(seqs) == 0:
            # Not enough frames for one sequence
            continue

        y = label_encoder.transform([gloss])[0]

        # Extract features for each sequence
        for seq in seqs:
            feat = extract_resnet_features(seq, resnet_model, transform, device)
            features_list.append(feat)
            labels_list.append(y)

    return np.array(features_list), np.array(labels_list)

C:\Users\Hayder\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\Hayder\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [15]:
print("Extracting train features...")
train_features, train_labels_encoded = process_split(train_folder)
print("Extracting val features...")
val_features, val_labels_encoded = process_split(val_folder)
print("Extracting test features...")
test_features, test_labels_encoded = process_split(test_folder)

Extracting train features...
Extracting val features...
Extracting test features...


In [16]:
############################################################
# Step 5: Create PyTorch Dataset and Dataloader
############################################################

class SignDataset(Dataset):
    def __init__(self, features, labels):
        # features shape: (N, seq_length, 2048)
        self.features = features
        self.labels = labels
    def __len__(self):
        return len(self.features)
    def __getitem__(self, idx):
        x = self.features[idx]  # (seq_length, 2048)
        y = self.labels[idx]
        x = torch.tensor(x, dtype=torch.float32)  # (seq_length,2048)
        y = torch.tensor(y, dtype=torch.long)
        return x, y


In [17]:
train_dataset = SignDataset(train_features, train_labels_encoded)
val_dataset = SignDataset(val_features, val_labels_encoded)
test_dataset = SignDataset(test_features, test_labels_encoded)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [22]:
np.save('train_labels_encoded_temp.npy', train_labels_encoded)
np.save('val_labels_encoded_temp.npy', val_labels_encoded)
np.save('test_labels_encoded_temp.npy', test_labels_encoded)

np.save('train_features_temp.npy', train_features)
np.save('val_features_temp.npy', val_features)
np.save('test_features_temp.npy', test_features)


In [18]:
############################################################
# Step 6: Define the Model
############################################################
class SignLanguageLSTM(nn.Module):
    def __init__(self, input_size=2048, hidden_size=512, num_classes=100, num_layers=2, dropout=0.5):
        super(SignLanguageLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, bidirectional=True)
        self.layer_norm = nn.LayerNorm(hidden_size*2)
        self.fc1 = nn.Linear(hidden_size*2, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, num_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x shape: (B, seq_length, 2048)
        lstm_out, _ = self.lstm(x) # (B, seq_length, hidden*2)
        lstm_out = self.layer_norm(lstm_out)
        out = torch.mean(lstm_out, dim=1)
        out = self.dropout(out)
        out = F.relu(self.fc1(out))
        out = self.dropout(out)
        out = F.relu(self.fc2(out))
        out = self.fc3(out)
        return out


In [19]:
model = SignLanguageLSTM(num_classes=len(label_encoder.classes_))
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)


In [20]:
############################################################
# Step 7: Train the model
############################################################

best_val_loss = float('inf')
num_epochs = 20
patience = 5
epochs_without_improvement = 0

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for features, labels in train_loader:
        features, labels = features.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(features)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()*features.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted==labels).sum().item()
        total += labels.size(0)
    train_loss = running_loss/total
    train_acc = correct/total

    model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for features, labels in val_loader:
            features, labels = features.to(device), labels.to(device)
            outputs = model(features)
            v_loss = criterion(outputs, labels)
            val_loss_sum += v_loss.item()*features.size(0)
            _, vpred = torch.max(outputs, 1)
            val_correct += (vpred==labels).sum().item()
            val_total += labels.size(0)
    val_loss = val_loss_sum/val_total
    val_acc = val_correct/val_total

    print(f"Epoch [{epoch+1}/{num_epochs}] | Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_without_improvement = 0
        # save best model if needed
        # torch.save(model.state_dict(), 'best_model.pth')
    # else:
    #     epochs_without_improvement += 1
    #     if epochs_without_improvement >= patience:
    #         print("Early stopping triggered.")
    #         break

Epoch [1/20] | Train Loss: 4.2017, Acc: 0.0891 | Val Loss: 3.6176, Acc: 0.2746
Epoch [2/20] | Train Loss: 2.1437, Acc: 0.5103 | Val Loss: 2.3568, Acc: 0.6099
Epoch [3/20] | Train Loss: 0.6990, Acc: 0.8444 | Val Loss: 2.3697, Acc: 0.6919
Epoch [4/20] | Train Loss: 0.3053, Acc: 0.9330 | Val Loss: 2.6406, Acc: 0.7012
Epoch [5/20] | Train Loss: 0.1921, Acc: 0.9578 | Val Loss: 2.8149, Acc: 0.7096
Epoch [6/20] | Train Loss: 0.1226, Acc: 0.9731 | Val Loss: 2.8907, Acc: 0.7070
Epoch [7/20] | Train Loss: 0.1405, Acc: 0.9655 | Val Loss: 3.0164, Acc: 0.7120
Epoch [8/20] | Train Loss: 0.0951, Acc: 0.9784 | Val Loss: 3.0396, Acc: 0.7100
Epoch [9/20] | Train Loss: 0.0740, Acc: 0.9824 | Val Loss: 3.3667, Acc: 0.7124
Epoch [10/20] | Train Loss: 0.1028, Acc: 0.9742 | Val Loss: 3.2793, Acc: 0.7117
Epoch [11/20] | Train Loss: 0.0901, Acc: 0.9773 | Val Loss: 3.4830, Acc: 0.7236
Epoch [12/20] | Train Loss: 0.0725, Acc: 0.9821 | Val Loss: 3.3532, Acc: 0.7204
Epoch [13/20] | Train Loss: 0.0614, Acc: 0.9844 |

In [21]:
############################################################
# Step 8: Evaluate on test set
############################################################

model.eval()
test_correct = 0
test_total = 0
with torch.no_grad():
    for features, labels in test_loader:
        features, labels = features.to(device), labels.to(device)
        outputs = model(features)
        _, preds = torch.max(outputs, 1)
        test_correct += (preds==labels).sum().item()
        test_total += labels.size(0)

test_acc = test_correct/test_total
print(f"Test Accuracy: {test_acc*100:.2f}%")

Test Accuracy: 68.03%
